In [1]:
# %pip install pandas numpy scikit-learn matplotlib tensorflow optuna

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from scipy import stats
import optuna
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

d:\NIHUT NAMAZ\Airfoil\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ------------------ Load Dataset ------------------
df = pd.read_excel("combined Assymetric.xlsx")

In [ ]:
# Input and Output Features
X = df[['m', 'p', 't', 'Mach', 're', 'AoA']]   # Input Features
y = df[['Cl', 'Cd', 'Cm']]                     # Output Features

In [4]:
# ------------------ Standardize Features ------------------
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [6]:
# ------------------ Optuna Objective ------------------
def objective(trial):
    # Train/Val/Test split suggestion
    train_ratio = trial.suggest_float("train_ratio", 0.6, 0.9)
    val_test_ratio = (1 - train_ratio) / 2

    # First split: Train vs Temp
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=1 - train_ratio, random_state=0)

    # Second split: Validation vs Test (from temp)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=0)

    # DNN hyperparameters
    n_layers = trial.suggest_int("n_layers", 1, 3)
    units = trial.suggest_int("units", 16, 128)
    activation = trial.suggest_categorical("activation", ["relu", "tanh"])
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)

    # Build the DNN
    model = Sequential()
    model.add(Dense(units, activation=activation, input_shape=(X_train.shape[1],)))
    for _ in range(n_layers - 1):
        model.add(Dense(units, activation=activation))
    model.add(Dense(3))  # Output layer

    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')

    # Train with early stopping using validation set
    model.fit(X_train, y_train,
              validation_data=(X_val, y_val),
              epochs=100,
              batch_size=32,
              verbose=0,
              callbacks=[
                  tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
              ])

    # Final evaluation on test set
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    return mse


In [7]:
# ------------------ Run Optuna Study ------------------
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

[I 2025-05-24 02:27:53,762] A new study created in memory with name: no-name-c329aa7f-83ff-4d40-b3de-8ff676153711
d:\NIHUT NAMAZ\Airfoil\.venv\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1062/1062 ━━━━━━━━━━━━━━━━━━━━ 1s 604us/step


[I 2025-05-24 02:35:09,648] Trial 0 finished with value: 0.00022705021547153592 and parameters: {'train_ratio': 0.6948320429118596, 'n_layers': 3, 'units': 52, 'activation': 'tanh', 'learning_rate': 0.00035317717805304083}. Best is trial 0 with value: 0.00022705021547153592.
d:\NIHUT NAMAZ\Airfoil\.venv\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1337/1337 ━━━━━━━━━━━━━━━━━━━━ 1s 562us/step


[I 2025-05-24 02:38:59,729] Trial 1 finished with value: 0.0006006645853631198 and parameters: {'train_ratio': 0.6156048548472445, 'n_layers': 3, 'units': 101, 'activation': 'relu', 'learning_rate': 0.008196939096331186}. Best is trial 0 with value: 0.00022705021547153592.
d:\NIHUT NAMAZ\Airfoil\.venv\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


...

[I 2025-05-24 12:29:52,716] Trial 17 finished with value: 0.00029513248591683805 and parameters: {'train_ratio': 0.7996144880077697, 'n_layers': 3, 'units': 56, 'activation': 'tanh', 'learning_rate': 0.0008286119602004335}. Best is trial 16 with value: 0.0002099958946928382.
d:\NIHUT NAMAZ\Airfoil\.venv\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1009/1009 ━━━━━━━━━━━━━━━━━━━━ 1s 561us/step


[I 2025-05-24 12:38:02,903] Trial 18 finished with value: 0.0003521881008055061 and parameters: {'train_ratio': 0.7100210113018925, 'n_layers': 2, 'units': 73, 'activation': 'tanh', 'learning_rate': 0.00023449570049099194}. Best is trial 16 with value: 0.0002099958946928382.
d:\NIHUT NAMAZ\Airfoil\.venv\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1201/1201 ━━━━━━━━━━━━━━━━━━━━ 1s 609us/step


[I 2025-05-24 12:45:01,235] Trial 19 finished with value: 0.00034172472078353167 and parameters: {'train_ratio': 0.6549089494634568, 'n_layers': 3, 'units': 19, 'activation': 'tanh', 'learning_rate': 0.0004818718618081335}. Best is trial 16 with value: 0.0002099958946928382.


In [8]:
# ------------------ Best Results ------------------
print("Best Parameters Found:")
print(f"Train Ratio:       {study.best_params['train_ratio']:.2f}")
print(f"Hidden Layers:     {study.best_params['n_layers']}")
print(f"Units per Layer:   {study.best_params['units']}")
print(f"Activation:        {study.best_params['activation']}")
print(f"Learning Rate:     {study.best_params['learning_rate']:.5f}")
print(f"Best Test MSE:     {study.best_value:.5f}")

Best Parameters Found:
Train Ratio:       0.70
Hidden Layers:     3
Units per Layer:   81
Activation:        tanh
Learning Rate:     0.00025
Best Test MSE:     0.00021


In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor

def rf_objective(trial):
    train_ratio = trial.suggest_float("train_ratio", 0.6, 0.9)
    val_test_ratio = (1 - train_ratio) / 2

    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=1 - train_ratio, random_state=0)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=0)

    # Hyperparameters for RF
    n_estimators = trial.suggest_int("n_estimators", 50, 200)
    max_depth = trial.suggest_int("max_depth", 5, 30)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2"])

    base_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_features=max_features,
        random_state=0,
        n_jobs=-1
    )

    model = MultiOutputRegressor(base_model)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    return mse

study_rf = optuna.create_study(direction="minimize")
study_rf.optimize(rf_objective, n_trials=20)

print("Best RF Parameters:")
print(study_rf.best_params)
print(f"Best RF Test MSE: {study_rf.best_value:.5f}")

[I 2025-05-26 04:50:31,771] A new study created in memory with name: no-name-42f6d07b-e2ac-415c-8f2a-d27e742d6f56
[I 2025-05-26 04:50:38,616] Trial 0 finished with value: 9.684714197145255e-05 and parameters: {'train_ratio': 0.6011231941968218, 'n_estimators': 121, 'max_depth': 29, 'max_features': 'sqrt'}. Best is trial 0 with value: 9.684714197145255e-05.
[I 2025-05-26 04:50:45,500] Trial 1 finished with value: 9.104328038671702e-05 and parameters: {'train_ratio': 0.7203067478203999, 'n_estimators': 171, 'max_depth': 19, 'max_features': 'log2'}. Best is trial 1 with value: 9.104328038671702e-05.
[I 2025-05-26 04:50:55,264] Trial 2 finished with value: 7.177420723949355e-05 and parameters: {'train_ratio': 0.7659769893808348, 'n_estimators': 154, 'max_depth': 28, 'max_features': 'log2'}. Best is trial 2 with value: 7.177420723949355e-05.
[I 2025-05-26 04:51:07,351] Trial 3 finished with value: 8.106064234964133e-05 and parameters: {'train_ratio': 0.8022538348701346, 'n_estimators': 182,

Best RF Parameters:
{'train_ratio': 0.7659769893808348, 'n_estimators': 154, 'max_depth': 28, 'max_features': 'log2'}
Best RF Test MSE: 0.00007


In [7]:
from sklearn.ensemble import GradientBoostingRegressor

def gb_objective(trial):
    train_ratio = trial.suggest_float("train_ratio", 0.6, 0.9)
    val_test_ratio = (1 - train_ratio) / 2

    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=1 - train_ratio, random_state=0)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=0)

    # Hyperparameters for GB
    n_estimators = trial.suggest_int("n_estimators", 50, 200)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3)
    max_depth = trial.suggest_int("max_depth", 3, 10)

    base_model = GradientBoostingRegressor(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        random_state=0
    )

    model = MultiOutputRegressor(base_model)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    return mse

study_gb = optuna.create_study(direction="minimize")
study_gb.optimize(gb_objective, n_trials=20)

print("Best GB Parameters:")
print(study_gb.best_params)
print(f"Best GB Test MSE: {study_gb.best_value:.5f}")


[I 2025-05-26 04:53:37,146] A new study created in memory with name: no-name-4a89a409-25c2-48b4-ae5a-81a851d650fa
[I 2025-05-26 04:54:16,351] Trial 0 finished with value: 0.0005348119807460985 and parameters: {'train_ratio': 0.7842317321195085, 'n_estimators': 83, 'learning_rate': 0.12327233381378182, 'max_depth': 6}. Best is trial 0 with value: 0.0005348119807460985.
[I 2025-05-26 08:56:47,471] Trial 1 finished with value: 0.00036506061110792523 and parameters: {'train_ratio': 0.7396789694565856, 'n_estimators': 162, 'learning_rate': 0.0352434810158378, 'max_depth': 8}. Best is trial 1 with value: 0.00036506061110792523.
[I 2025-05-26 08:57:40,433] Trial 2 finished with value: 0.002022253730459357 and parameters: {'train_ratio': 0.6211564698375279, 'n_estimators': 168, 'learning_rate': 0.018278238714920462, 'max_depth': 5}. Best is trial 1 with value: 0.00036506061110792523.
[I 2025-05-26 08:59:05,251] Trial 3 finished with value: 0.00017059503401279825 and parameters: {'train_ratio':

Best GB Parameters:
{'train_ratio': 0.7167124213095897, 'n_estimators': 147, 'learning_rate': 0.21632794021182905, 'max_depth': 10}
Best GB Test MSE: 0.00009
